# Feature Engineering: Editions Dataset (Pandas)

This notebook performs data quality checks and feature engineering on the `editions.parquet` dataset using **Pandas**.

## Schema (Input)
- **edition_key** (String, nullable)
- **publish_date** (String, nullable)
- **work_key** (String, nullable)

## Objectives
1. Assess data quality (null values, format issues)
2. Extract `publish_year` from `publish_date` (various formats)
3. Validate `work_key` (required for joins; must start with `/works/`)
4. Remove rows with invalid or null `work_key`
5. Generate quality report

In [2]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# Configuration
DATA_DIR = Path('../data')
PROCESSED_DIR = DATA_DIR / 'processed'
REPORTS_DIR = Path('../reports')
REPORTS_DIR.mkdir(exist_ok=True)

## Step 1: Load and Inspect Raw Data

In [3]:
# Load editions data
input_path = PROCESSED_DIR / 'editions.parquet'
df = pd.read_parquet(input_path)

print(f"Total rows: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nSchema:")
print(df.dtypes)
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Total rows: 55,591,553
Columns: ['edition_key', 'publish_date', 'work_key']

Schema:
edition_key     str
publish_date    str
work_key        str
dtype: object

Memory usage: 3412.07 MB


In [4]:
# Display first few rows
print(df.head(10))
print(df.tail(10))

          edition_key        publish_date            work_key
0  /books/OL10000281M           July 1997  /works/OL14901213W
1  /books/OL10000294M    December 3, 1997  /works/OL14903225W
2  /books/OL10000463M     October 9, 1997  /works/OL14902902W
3  /books/OL10000827M        June 4, 1998  /works/OL14903289W
4  /books/OL10000975M  September 30, 1998  /works/OL14903171W
5  /books/OL10001082M       February 1999  /works/OL14892590W
6  /books/OL10001144M    January 20, 1999  /works/OL14903235W
7   /books/OL1000116M                1997   /works/OL3336680W
8  /books/OL10001360M        June 7, 1999  /works/OL14903129W
9  /books/OL10002012M    January 23, 1997  /works/OL14900685W
                edition_key       publish_date            work_key
55591543  /books/OL9998774M  December 31, 1996  /works/OL14903445W
55591544  /books/OL9999058M       May 19, 1995  /works/OL14903445W
55591545  /books/OL9999177M      June 19, 1999  /works/OL14903445W
55591546  /books/OL9999189M      June 19, 1999  /w

## Step 2: Data Quality Assessment

In [5]:
# Check null values
print("=== Null Value Counts ===")
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df)) * 100

quality_df = pd.DataFrame({
    'Column': null_counts.index,
    'Null Count': null_counts.values,
    'Null Percentage': null_pct.values
})
print(quality_df.to_string(index=False))

=== Null Value Counts ===
      Column  Null Count  Null Percentage
 edition_key           0         0.000000
publish_date     1718239         3.090827
    work_key     1959420         3.524672


In [6]:
# Sample publish_date values to understand format patterns
print("=== Sample publish_date Values ===")
sample_dates = df['publish_date'].dropna().head(50)
for i, date_val in enumerate(sample_dates, 1):
    print(f"{i:2d}. '{date_val}'")

=== Sample publish_date Values ===
 1. 'July 1997'
 2. 'December 3, 1997'
 3. 'October 9, 1997'
 4. 'June 4, 1998'
 5. 'September 30, 1998'
 6. 'February 1999'
 7. 'January 20, 1999'
 8. '1997'
 9. 'June 7, 1999'
10. 'January 23, 1997'
11. 'June 13, 1996'
12. 'February 20, 1996'
13. 'April 28, 1996'
14. 'June 14, 1995'
15. 'March 26, 1997'
16. '1965'
17. 'December 31, 1995'
18. 'April 30, 1995'
19. 'May 29, 1998'
20. 'May 15, 1996'
21. 'May 15, 1996'
22. 'April 30, 1995'
23. 'December 11, 1996'
24. 'March 12, 1998'
25. 'November 24, 1998'
26. 'December 31, 1996'
27. 'May 29, 1998'
28. 'June 12, 1998'
29. 'December 31, 1990'
30. 'December 31, 1990'
31. 'December 31, 1991'
32. 'December 31, 1991'
33. 'December 31, 1991'
34. 'December 31, 1991'
35. 'December 31, 1992'
36. 'December 31, 1992'
37. 'December 31, 1992'
38. 'December 31, 1992'
39. 'December 31, 1992'
40. 'December 31, 1992'
41. 'December 31, 1993'
42. 'December 31, 1992'
43. 'December 31, 1992'
44. 'December 31, 1992'
45. 'Dec

In [7]:
# Analyze publish_date format patterns
print("=== Publish Date Format Analysis ===")
non_null_dates = df['publish_date'].dropna()

if len(non_null_dates) > 0:
    patterns = {
        '4-digit year only': non_null_dates.str.match(r'^\d{4}$', na=False).sum(),
        'ISO date (YYYY-MM-DD)': non_null_dates.str.match(r'^\d{4}-\d{2}-\d{2}$', na=False).sum(),
        'Partial date (YYYY-MM)': non_null_dates.str.match(r'^\d{4}-\d{2}$', na=False).sum(),
        'Contains "c." or "ca."': non_null_dates.str.contains(r'c\.|ca\.', case=False, na=False).sum(),
        'Contains range (YYYY-YYYY)': non_null_dates.str.contains(r'\d{4}\s*-\s*\d{4}', na=False).sum(),
        'fl. (floruit)': non_null_dates.str.match(r'^fl\.?\s*\d{4}', na=False).sum(),
        'b. (born)': non_null_dates.str.match(r'^b\.?\s*\d{4}', na=False).sum(),
        'Year then month/day': non_null_dates.str.match(r'^\d{4}\s+[A-Za-z]', na=False).sum(),
        'Year in parentheses or brackets': non_null_dates.str.match(r'^[\(\[]\s*\d{4}', na=False).sum(),
        "Decade (1850s or 1850's)": non_null_dates.str.contains(r"\d{4}'?s\b", na=False).sum(),
        'circa / about / ~': non_null_dates.str.contains(r'circa|about|~', case=False, na=False).sum(),
        'est. / estimated': non_null_dates.str.contains(r'est\.?|estimated', case=False, na=False).sum(),
        "Year with trailing ? or .": non_null_dates.str.match(r"^\d{4}[\.\?]\s*$", na=False).sum(),
        'Slash or dot date': non_null_dates.str.contains(r'\d{4}[/\.]\d{1,2}[/\.]\d{1,2}|\d{1,2}[/\.]\d{1,2}[/\.]\d{4}', na=False).sum(),
        'Month name + year': non_null_dates.str.contains(r'(January|February|March|April|May|June|July|August|September|October|November|December|Jan|Feb|Mar|Apr|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[\.]?\s+\d{4}', case=False, na=False).sum(),
    }
    for pattern, count in patterns.items():
        pct = (count / len(non_null_dates)) * 100
        print(f"{pattern}: {count:,} ({pct:.2f}%)")
    has_four_digit_year = non_null_dates.str.contains(r'\d{4}', na=False).sum()
    print(f"\nContains any 4-digit year: {has_four_digit_year:,} ({has_four_digit_year/len(non_null_dates)*100:.2f}%)")

=== Publish Date Format Analysis ===


/var/folders/9c/878pcq3j5vs8nslt0z5sj8380000gn/T/ipykernel_39906/273401019.py:21: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  'Month name + year': non_null_dates.str.contains(r'(January|February|March|April|May|June|July|August|September|October|November|December|Jan|Feb|Mar|Apr|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[\.]?\s+\d{4}', case=False, na=False).sum(),


4-digit year only: 44,592,746 (82.77%)
ISO date (YYYY-MM-DD): 1,257,295 (2.33%)
Partial date (YYYY-MM): 13,649 (0.03%)
Contains "c." or "ca.": 557 (0.00%)
Contains range (YYYY-YYYY): 2,483 (0.00%)
fl. (floruit): 0 (0.00%)
b. (born): 0 (0.00%)
Year then month/day: 2,646 (0.00%)
Year in parentheses or brackets: 1,789 (0.00%)
Decade (1850s or 1850's): 217 (0.00%)
circa / about / ~: 85 (0.00%)
est. / estimated: 88 (0.00%)
Year with trailing ? or .: 5,013 (0.01%)
Slash or dot date: 8,138 (0.02%)
Month name + year: 2,166,778 (4.02%)

Contains any 4-digit year: 53,780,246 (99.83%)


In [8]:
# Work_key validation: check format (should start with /works/)
print("=== work_key Validation ===")
work_key_non_null = df['work_key'].dropna()
valid_work_key = work_key_non_null.str.startswith('/works/', na=False)
print(f"Rows with non-null work_key: {len(work_key_non_null):,}")
print(f"Rows with work_key starting with /works/: {valid_work_key.sum():,}")
print(f"Rows with malformed work_key: {(~valid_work_key).sum():,}")

# Sample malformed work_key if any
malformed = work_key_non_null[~valid_work_key]
if len(malformed) > 0:
    print(f"\nSample malformed work_key values:")
    for val in malformed.head(10):
        print(f"  '{val}'")

=== work_key Validation ===
Rows with non-null work_key: 53,632,133
Rows with work_key starting with /works/: 53,632,133
Rows with malformed work_key: 0


## Step 3: Extract Publish Year from Various Formats

In [9]:
def extract_year_from_date(date_str):
    """
    Extract year from various date formats (same logic as authors notebook).
    Used for publish_date in editions.
    Returns: Integer year if extractable, None otherwise. Valid range: 0-2026.
    """
    if pd.isna(date_str) or date_str == "":
        return None
    date_str = str(date_str).strip()
    match = re.match(r'^(\d{4})[\.\?]?\s*$', date_str)
    if match:
        year = int(match.group(1))
        if 0 <= year <= 2026: return year
    match = re.match(r'^(\d{4})-\d{1,2}', date_str)
    if match:
        year = int(match.group(1))
        if 0 <= year <= 2026: return year
    match = re.search(r'c\.?\s*(\d{4})|ca\.?\s*(\d{4})|circa\s*(\d{4})|about\s*(\d{4})|~\s*(\d{4})|est\.?\s*(\d{4})|estimated\s*(\d{4})', date_str, re.IGNORECASE)
    if match:
        year = int(next(g for g in match.groups() if g is not None))
        if 0 <= year <= 2026: return year
    match = re.match(r'^(?:fl|b|d)\.?\s*(\d{4})', date_str, re.IGNORECASE)
    if match:
        year = int(match.group(1))
        if 0 <= year <= 2026: return year
    match = re.match(r'^(\d{4})\s+', date_str)
    if match:
        year = int(match.group(1))
        if 0 <= year <= 2026: return year
    match = re.match(r'^[\(\[]\s*(\d{4})[\)\]]?\s*', date_str)
    if match:
        year = int(match.group(1))
        if 0 <= year <= 2026: return year
    match = re.match(r'(\d{4})\s*-\s*\d{4}', date_str)
    if match:
        year = int(match.group(1))
        if 0 <= year <= 2026: return year
    match = re.match(r"^(\d{4})'?s\b", date_str, re.IGNORECASE)
    if match:
        year = int(match.group(1))
        if 0 <= year <= 2026: return year
    match = re.match(r'^(\d{4})[/\.]\d{1,2}[/\.]\d{1,2}', date_str)
    if match:
        year = int(match.group(1))
        if 0 <= year <= 2026: return year
    match = re.search(r'(\d{4})[/\.]\d{1,2}[/\.]\d{1,2}', date_str)
    if match:
        year = int(match.group(1))
        if 0 <= year <= 2026: return year
    match = re.search(r'(?:January|February|March|April|May|June|July|August|September|October|November|December|Jan|Feb|Mar|Apr|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[\.]?\s+(\d{4})\b', date_str, re.IGNORECASE)
    if match:
        year = int(match.group(1))
        if 0 <= year <= 2026: return year
    match = re.search(r'\b(\d{4})\b', date_str)
    if match:
        year = int(match.group(1))
        if 0 <= year <= 2026: return year
    match = re.search(r'(\d{4})', date_str)
    if match:
        year = int(match.group(1))
        if 0 <= year <= 2026: return year
    return None

In [10]:
# Test year extraction on sample publish_date values
test_dates = ["2020", "2020-05-15", "c. 1995", "1990s", "(2000", None, ""]
print("Testing year extraction:")
for d in test_dates:
    print(f"  '{d}' -> {extract_year_from_date(d)}")

Testing year extraction:
  '2020' -> 2020
  '2020-05-15' -> 2020
  'c. 1995' -> 1995
  '1990s' -> 1990
  '(2000' -> 2000
  'None' -> None
  '' -> None


## Step 4: Clean and Transform Data

In [11]:
df_cleaned = df.copy()
print(f"Original row count: {len(df_cleaned):,}")

Original row count: 55,591,553


In [12]:
# Extract publish_year from publish_date
print("Extracting publish_year from publish_date...")
df_cleaned['publish_year'] = df_cleaned['publish_date'].apply(extract_year_from_date)
df_cleaned['publish_year'] = df_cleaned['publish_year'].astype('Int16')
print(f"Rows with valid publish_year: {df_cleaned['publish_year'].notna().sum():,}")
print(f"Rows with null publish_year: {df_cleaned['publish_year'].isna().sum():,}")

Extracting publish_year from publish_date...
Rows with valid publish_year: 53,779,167
Rows with null publish_year: 1,812,386


In [13]:
# Keep only rows with valid work_key (required for joins)
# Valid: non-null and starts with /works/
before_filter = len(df_cleaned)
df_cleaned = df_cleaned[
    df_cleaned['work_key'].notna() &
    (df_cleaned['work_key'].str.strip() != '') &
    df_cleaned['work_key'].str.startswith('/works/', na=False)
].copy()
rows_removed = before_filter - len(df_cleaned)
print(f"Rows removed (invalid/null work_key): {rows_removed:,}")
print(f"Rows retained: {len(df_cleaned):,}")

Rows removed (invalid/null work_key): 1,959,420
Rows retained: 53,632,133


In [14]:
# Ensure edition_key is present (drop if null)
df_cleaned = df_cleaned[df_cleaned['edition_key'].notna()].copy()

# Select final columns: edition_key, publish_year, work_key
df_cleaned = df_cleaned[["edition_key", "publish_year", "work_key"]].copy()

print("Final schema:")
print(df_cleaned.dtypes)
print(f"\nFinal row count: {len(df_cleaned):,}")

Final schema:
edition_key       str
publish_year    Int16
work_key          str
dtype: object

Final row count: 53,632,133


## Step 5: Final Quality Check

In [15]:
print("=== Final Quality Check ===")
valid_publish_year = df_cleaned['publish_year'].notna().sum()
pct_valid = (valid_publish_year / len(df_cleaned)) * 100 if len(df_cleaned) > 0 else 0
print(f"Rows with valid publish_year: {valid_publish_year:,} ({pct_valid:.2f}%)")
print(f"Rows with null publish_year: {len(df_cleaned) - valid_publish_year:,}")
if valid_publish_year > 0:
    print(f"\nPublish year - Min: {df_cleaned['publish_year'].min()}, Max: {df_cleaned['publish_year'].max()}, Median: {df_cleaned['publish_year'].median():.0f}")

=== Final Quality Check ===
Rows with valid publish_year: 52,005,128 (96.97%)
Rows with null publish_year: 1,627,005

Publish year - Min: 0, Max: 2026, Median: 2002


In [16]:
print("\nSample of cleaned data:")
df_cleaned.head(20)


Sample of cleaned data:


,edition_key,publish_year,work_key
0,/books/OL10000281M,1997,/works/OL14901213W
1,/books/OL10000294M,1997,/works/OL14903225W
2,/books/OL10000463M,1997,/works/OL14902902W
3,/books/OL10000827M,1998,/works/OL14903289W
4,/books/OL10000975M,1998,/works/OL14903171W
5,/books/OL10001082M,1999,/works/OL14892590W
6,/books/OL10001144M,1999,/works/OL14903235W
7,/books/OL1000116M,1997,/works/OL3336680W
8,/books/OL10001360M,1999,/works/OL14903129W
9,/books/OL10002012M,1997,/works/OL14900685W


## Step 6: Save Cleaned Data

In [17]:
output_path = PROCESSED_DIR / 'editions_cleaned.parquet'
df_cleaned.to_parquet(output_path, index=False)
print(f"Saved cleaned data to {output_path}")
print(f"File size: {output_path.stat().st_size / 1024**2:.2f} MB")

Saved cleaned data to ../data/processed/editions_cleaned.parquet
File size: 775.69 MB


## Step 7: Generate Quality Report

In [18]:
report_path = REPORTS_DIR / 'data_quality_editions_pandas.md'
valid_count = df_cleaned['publish_year'].notna().sum()
null_count = df_cleaned['publish_year'].isna().sum()

report = f"""# Data Quality Report: Editions Dataset (Pandas)

## Summary
- **Original row count**: {len(df):,}
- **Cleaned row count**: {len(df_cleaned):,}
- **Rows removed**: {len(df) - len(df_cleaned):,} ({(len(df) - len(df_cleaned))/len(df)*100:.2f}%)
- **Rows retained**: {len(df_cleaned)/len(df)*100:.2f}%

## Data Quality Metrics

### Publish Year Extraction
- **Rows with valid publish_year**: {valid_count:,} ({valid_count/len(df_cleaned)*100:.2f}%)
- **Rows with null publish_year**: {null_count:,} ({null_count/len(df_cleaned)*100:.2f}%)

"""
if valid_count > 0:
    report += f"""### Publish Year Statistics
- **Minimum year**: {df_cleaned['publish_year'].min()}
- **Maximum year**: {df_cleaned['publish_year'].max()}
- **Median year**: {df_cleaned['publish_year'].median():.0f}

"""
report += """## Schema Changes
- **Removed**: `publish_date` (string)
- **Added**: `publish_year` (Int16, nullable)

## Cleaning Steps Applied
1. Extracted year from `publish_date` using regex patterns (same as authors)
2. Validated years (0-2026)
3. Removed rows with null or invalid `work_key` (must start with /works/)
4. Removed rows with null `edition_key`
"""
report_path.write_text(report)
print(f"Quality report saved to {report_path}")
print("\n" + report_path.read_text())

Quality report saved to ../reports/data_quality_editions_pandas.md

# Data Quality Report: Editions Dataset (Pandas)

## Summary
- **Original row count**: 55,591,553
- **Cleaned row count**: 53,632,133
- **Rows removed**: 1,959,420 (3.52%)
- **Rows retained**: 96.48%

## Data Quality Metrics

### Publish Year Extraction
- **Rows with valid publish_year**: 52,005,128 (96.97%)
- **Rows with null publish_year**: 1,627,005 (3.03%)

### Publish Year Statistics
- **Minimum year**: 0
- **Maximum year**: 2026
- **Median year**: 2002

## Schema Changes
- **Removed**: `publish_date` (string)
- **Added**: `publish_year` (Int16, nullable)

## Cleaning Steps Applied
1. Extracted year from `publish_date` using regex patterns (same as authors)
2. Validated years (0-2026)
3. Removed rows with null or invalid `work_key` (must start with /works/)
4. Removed rows with null `edition_key`



## Summary

Done: load & inspect → quality assessment → publish_year extraction → work_key validation → save `editions_cleaned.parquet` and quality report.